In [1]:
!pip3 install Pillow reportlab matplotlib pyodbc

In [2]:
import os
import random
from PIL import Image, ImageDraw, ImageFont, ImageEnhance
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
import io

def generate_coupon_pdf(
    coupon_image_path,
    output_pdf_path,
    rows_per_page=3,
    cols_per_page=2,
    coupon_code_pattern="A{number:04d}",
    start_number=1,
    end_number=9999,
    dpi=300,  # High DPI for better quality
    jpeg_quality=98,  # Maximum JPEG quality
    upscale_factor=2.0,  # Upscale images before processing
    preserve_original_size=False  # Option to maintain original image dimensions
):
    try:
        # Load with high quality settings
        original_image = Image.open(coupon_image_path)
        if original_image.mode != 'RGBA':
            original_image = original_image.convert('RGBA')

        # Enhance image quality before processing
        enhancer = ImageEnhance.Sharpness(original_image)
        original_image = enhancer.enhance(1.1)  # Slight sharpness boost

        print(f"✅ Loaded coupon image: {coupon_image_path}")
        print(f"📐 Original size: {original_image.size[0]}x{original_image.size[1]} pixels")

        # Upscale original image for better quality
        if upscale_factor > 1.0:
            new_size = (
                int(original_image.size[0] * upscale_factor),
                int(original_image.size[1] * upscale_factor)
            )
            original_image = original_image.resize(new_size, Image.Resampling.LANCZOS)
            print(f"🔍 Upscaled to: {original_image.size[0]}x{original_image.size[1]} pixels")

    except Exception as e:
        print(f"❌ Error loading image: {e}")
        return

    rows = rows_per_page
    cols = cols_per_page
    coupons_per_page = rows * cols

    print(f"📄 Layout: {cols} columns x {rows} rows = {coupons_per_page} coupons per page")

    # Convert DPI to points (1 inch = 72 points)
    page_width, page_height = A4
    margin = 2  # Increased margin for better layout
    gutter = 1  # Increased gutter for cleaner separation

    available_width = page_width - (2 * margin)
    available_height = page_height - (2 * margin)

    total_gutter_width = gutter * (cols - 1) if cols > 1 else 0
    total_gutter_height = gutter * (rows - 1) if rows > 1 else 0

    coupon_width = (available_width - total_gutter_width) / cols
    coupon_height = (available_height - total_gutter_height) / rows

    if preserve_original_size:
        # Calculate based on original image proportions at high DPI
        original_width, original_height = original_image.size
        # Convert pixels to points at specified DPI
        original_width_points = (original_width * 72) / dpi
        original_height_points = (original_height * 72) / dpi

        # Use smaller dimension to fit within available space
        scale_factor = min(coupon_width / original_width_points, coupon_height / original_height_points)
        coupon_width = original_width_points * scale_factor
        coupon_height = original_height_points * scale_factor
    else:
        # Maintain aspect ratio
        original_width, original_height = original_image.size
        aspect_ratio = original_width / original_height

        if coupon_width / coupon_height > aspect_ratio:
            coupon_width = coupon_height * aspect_ratio
        else:
            coupon_height = coupon_width / aspect_ratio

    print(f"📏 Each coupon: {coupon_width:.1f} x {coupon_height:.1f} points")
    print(f"🔲 Gutter spacing: {gutter} points between coupons")
    print(f"🎯 Target DPI: {dpi}")
    print(f"📊 Page utilization: {((available_width * available_height) / (page_width * page_height) * 100):.1f}%")

    # Sequential numbers instead of random
    all_numbers = list(range(start_number, end_number + 1))

    def add_coupon_code_to_image(image, coupon_code):
        if image.mode != 'RGBA':
            image = image.convert('RGBA')

        img_with_code = image.copy()
        draw = ImageDraw.Draw(img_with_code)

        img_width, img_height = img_with_code.size

        # Scale font size based on image resolution
        base_font_size = max(36, int(min(img_width, img_height) * 0.08))  # Larger base font
        font_size = int(base_font_size * upscale_factor)

        font = None
        font_paths = [
            "arialbd.ttf", "Arial-Bold.ttf", "ARIALBD.TTF",
            "arial.ttf", "Arial.ttf", "ARIAL.TTF",
            "helvetica-bold.ttf", "Helvetica-Bold.ttf", "HELVETICA-BOLD.TTF",
            "helvetica.ttf", "Helvetica.ttf", "HELVETICA.TTF",
            "calibrib.ttf", "Calibri-Bold.ttf", "CALIBRIB.TTF",
            "DejaVuSans-Bold.ttf", "dejavu-sans-bold.ttf",
            "DejaVuSans.ttf", "dejavu-sans.ttf",
            "calibri.ttf", "Calibri.ttf", "CALIBRI.TTF",
            "segoeui.ttf", "SegoeUI.ttf",
            "tahoma.ttf", "Tahoma.ttf"
        ]

        for font_path in font_paths:
            try:
                font = ImageFont.truetype(font_path, font_size)
                print(f"   Using font: {font_path} at {font_size}px")
                break
            except:
                continue

        if font is None:
            try:
                font = ImageFont.load_default()
                print(f"   Using default font at {font_size}px")
            except:
                print("   Warning: Could not load any font!")
                return img_with_code

        try:
            bbox = draw.textbbox((0, 0), coupon_code, font=font)
            text_width = bbox[2] - bbox[0]
            text_height = bbox[3] - bbox[1]
        except:
            text_width, text_height = font.getsize(coupon_code)

        # Position text at bottom center of the image
        x = (img_width - text_width) // 2
        y = img_height - text_height - max(40, int(img_height * 0.06))  # Better positioning

        if x < 10:
            x = 10
        if x + text_width > img_width - 10:
            x = img_width - text_width - 10
        if y < 10:
            y = 10

        # Draw text directly on the image with black color - NO BACKGROUND
        # Multiple passes for bold effect
        for offset_x in range(3):
            for offset_y in range(3):
                draw.text((x + offset_x, y + offset_y), coupon_code, fill=(0, 0, 0, 255), font=font)

        print(f"   Added code '{coupon_code}' at position ({x}, {y}) directly on image")
        return img_with_code

    try:
        c = canvas.Canvas(output_pdf_path, pagesize=A4)
        coupon_count = 0
        page_count = 1

        print(f"🎫 Generating high-quality coupons with pattern: {coupon_code_pattern}")
        print(f"🔢 Sequential order from {start_number} to {end_number}")

        while coupon_count < len(all_numbers):
            print(f"📄 Creating page {page_count}...")

            for row in range(rows):
                for col in range(cols):
                    if coupon_count >= len(all_numbers):
                        break

                    number = all_numbers[coupon_count]
                    coupon_code = coupon_code_pattern.format(number=number)

                    print(f"   Processing coupon {coupon_count + 1}: {coupon_code}")

                    coupon_with_code = add_coupon_code_to_image(original_image, coupon_code)

                    # Calculate target size in pixels based on DPI
                    target_width_px = int((coupon_width * dpi) / 72)
                    target_height_px = int((coupon_height * dpi) / 72)

                    # High-quality resize with better resampling
                    if coupon_with_code.size != (target_width_px, target_height_px):
                        # Use best quality resampling method
                        coupon_resized = coupon_with_code.resize(
                            (target_width_px, target_height_px),
                            Image.Resampling.LANCZOS
                        )

                        # Apply slight sharpening after resize
                        enhancer = ImageEnhance.Sharpness(coupon_resized)
                        coupon_resized = enhancer.enhance(1.05)
                    else:
                        coupon_resized = coupon_with_code

                    x = margin + col * (coupon_width + gutter)
                    y = page_height - margin - (row + 1) * coupon_height - row * gutter

                    # Convert RGBA to RGB for JPEG (composite on white background for PDF)
                    if coupon_resized.mode == 'RGBA':
                        # Create white background for PDF
                        rgb_image = Image.new('RGB', coupon_resized.size, (255, 255, 255))
                        rgb_image.paste(coupon_resized, mask=coupon_resized.split()[-1])  # Use alpha channel as mask
                        coupon_resized = rgb_image

                    # Use JPEG with maximum quality for better compression/quality balance
                    img_buffer = io.BytesIO()
                    coupon_resized.save(
                        img_buffer,
                        format='JPEG',
                        quality=jpeg_quality,
                        optimize=True,
                        progressive=True,  # Progressive JPEG for better quality
                        subsampling=0,     # No chroma subsampling for best quality
                        qtables='web_high'  # High quality quantization tables
                    )
                    img_buffer.seek(0)

                    img_reader = ImageReader(img_buffer)
                    c.drawImage(
                        img_reader,
                        x, y,
                        width=coupon_width,
                        height=coupon_height,
                        preserveAspectRatio=True,
                        mask='auto'
                    )

                    coupon_count += 1

                if coupon_count >= len(all_numbers):
                    break

            if coupon_count < len(all_numbers):
                c.showPage()
                page_count += 1

        c.save()
        print(f"✅ High-quality PDF created successfully!")
        print(f"📁 File: {output_pdf_path}")
        print(f"🎫 Total coupons: {coupon_count}")
        print(f"📄 Total pages: {page_count}")
        print(f"🎯 Quality settings: {dpi} DPI, {jpeg_quality}% JPEG quality, {upscale_factor}x upscaling")

        first_code = coupon_code_pattern.format(number=all_numbers[0])
        last_code = coupon_code_pattern.format(number=all_numbers[min(coupon_count-1, len(all_numbers)-1)])
        print(f"🏷️  Coupon codes: {first_code} to {last_code} (sequential order)")

    except Exception as e:
        print(f"❌ Error creating PDF: {e}")

def main():
    print("🎫 Welcome to HIGH-QUALITY Coupon Generator!")
    print("=" * 55)

    while True:
        coupon_image = input("\n📁 Enter path to your coupon image file: ").strip()
        if os.path.exists(coupon_image):
            print(f"✅ Image found: {coupon_image}")
            break
        else:
            print(f"❌ File not found: {coupon_image}")
            print("Please check the path and try again.")

    output_file = input("\n💾 Enter output PDF filename (default: high_quality_coupons.pdf): ").strip()
    if not output_file:
        output_file = "high_quality_coupons.pdf"

    while True:
        try:
            rows = int(input(f"\n📐 How many ROWS of coupons per page? (1-20): "))
            if 1 <= rows <= 20:
                break
            else:
                print("Please enter a number between 1 and 20.")
        except ValueError:
            print("Please enter a valid number.")

    while True:
        try:
            cols = int(input(f"📐 How many COLUMNS of coupons per page? (1-20): "))
            if 1 <= cols <= 20:
                break
            else:
                print("Please enter a number between 1 and 20.")
        except ValueError:
            print("Please enter a valid number.")

    total_per_page = rows * cols
    print(f"✅ Layout: {cols} columns × {rows} rows = {total_per_page} coupons per page")

    print(f"\n🏷️  Coupon Code Pattern Examples:")
    print("   A{number:04d} → A0001, A0002, A0003...")
    print("   SAVE{number} → SAVE1, SAVE2, SAVE3...")
    print("   COUPON-{number:05d} → COUPON-00001, COUPON-00002...")

    pattern = input("Enter your pattern (default: A{number:04d}): ").strip()
    if not pattern:
        pattern = "A{number:04d}"

    while True:
        try:
            start_num = int(input(f"\n🔢 Starting number (default: 1): ") or "1")
            break
        except ValueError:
            print("Please enter a valid number.")

    while True:
        try:
            default_end = start_num + (total_per_page * 10) - 1
            end_num = int(input(f"🔢 Ending number (default: {default_end}): ") or str(default_end))
            if end_num >= start_num:
                break
            else:
                print("Ending number must be greater than or equal to starting number.")
        except ValueError:
            print("Please enter a valid number.")

    total_codes = end_num - start_num + 1
    estimated_pages = (total_codes + total_per_page - 1) // total_per_page

    print(f"\n📋 Summary:")
    print(f"   Image: {coupon_image}")
    print(f"   Output: {output_file}")
    print(f"   Layout: {cols} columns × {rows} rows = {total_per_page} coupons per page")
    print(f"   Pattern: {pattern}")
    print(f"   Range: {start_num} to {end_num} ({total_codes} total codes)")
    print(f"   Estimated pages: {estimated_pages}")
    print(f"   Order: Sequential (not random)")

    
    generate_coupon_pdf(
        coupon_image_path=coupon_image,
        output_pdf_path=output_file,
        rows_per_page=rows,
        cols_per_page=cols,
        coupon_code_pattern=pattern,
        start_number=start_num,
        end_number=end_num
    )
    

def quick_demo_hq():
    """High-quality demo version"""
    coupon_image = "coupon.jpg"
    output_file = "Tokens.pdf"

    if not os.path.exists(coupon_image):
        print(f"⚠️  Please save your coupon image as '{coupon_image}' or update the path")
        return

    generate_coupon_pdf(
        coupon_image_path=coupon_image,
        output_pdf_path=output_file,
        rows_per_page=3,
        cols_per_page=2,
        coupon_code_pattern="A{number:04d}",
        start_number=1,
        end_number=100
    )

print("🎫 HIGH-QUALITY Coupon Generator Ready!")
print("\n🚀 NEW QUALITY FEATURES:")
print("✅ Configurable DPI (300+ recommended)")
print("✅ Image upscaling before processing")
print("✅ Maximum JPEG quality (98%)")
print("✅ Progressive JPEG with optimal settings")
print("✅ Image sharpening enhancements")
print("✅ Anti-aliased text rendering")
print("✅ Better font scaling for high-res")
print("✅ TEXT DIRECTLY ON IMAGE (no background)")
print("✅ SEQUENTIAL coupon numbering")

print("\nOptions:")
print("1. Run main() - Interactive mode with quality controls")
print("2. Run quick_demo_hq() - High-quality demo")
print("3. Call generate_coupon_pdf() with custom parameters")

print("\nQuality Tips:")
print("🎯 Use DPI 300+ for print quality")
print("🔍 Higher upscale factor = sharper images")
print("📐 Lower coupons per page = larger, clearer coupons")
print("🖼️  Start with highest resolution source image")

print("\nRun main() to get started with HIGH-QUALITY output!")

main()

🎫 HIGH-QUALITY Coupon Generator Ready!

🚀 NEW QUALITY FEATURES:
✅ Configurable DPI (300+ recommended)
✅ Image upscaling before processing
✅ Maximum JPEG quality (98%)
✅ Progressive JPEG with optimal settings
✅ Image sharpening enhancements
✅ Anti-aliased text rendering
✅ Better font scaling for high-res
✅ TEXT DIRECTLY ON IMAGE (no background)
✅ SEQUENTIAL coupon numbering

Options:
1. Run main() - Interactive mode with quality controls
2. Run quick_demo_hq() - High-quality demo
3. Call generate_coupon_pdf() with custom parameters

Quality Tips:
🎯 Use DPI 300+ for print quality
🔍 Higher upscale factor = sharper images
📐 Lower coupons per page = larger, clearer coupons
🖼️  Start with highest resolution source image

Run main() to get started with HIGH-QUALITY output!
🎫 Welcome to HIGH-QUALITY Coupon Generator!
✅ Image found: /Users/shivanisinghal/Downloads/Cera Power Token/Havit_Token.png
✅ Layout: 3 columns × 6 rows = 18 coupons per page

🏷️  Coupon Code Pattern Examples:
   A{number:0